# Deadline markets — prices and individual trades

This downloads the **seven audited contracts** from notebook 03: Hormuz September–December and Russia–Ukraine October–December. No matching September 2026 Ukraine agreement contract was found; we do not substitute a different question.

**What “finest data” means**

- Keep every returned **individual taker-side fill**, with its original **blockchain timestamp in seconds**. These are not millisecond exchange/quote timestamps; fills in the same second do not establish a tradable lead–lag.
- Price-history granularity starts at **60 seconds**, not one second. Retention depends on resolution: the API documents at least 7 days at one minute, 60 days at five minutes, 90 days at thirty minutes, and permanent three-hour history. We query overlapping tiers and save the **actual** resolution of each observation.
- Retrieve each contract's lifetime available history, not just the common window. Later analysis must restrict comparisons to overlapping observed periods.
- No fake prices, interpolation, orders, wallet analysis, or external uploads. Raw page records and all output stay under Git-ignored `data/`.

**Review note:** the committed version of notebook 03 currently exports six October-December contracts, but this downloader requires seven, including September Hormuz. These notebooks are shared for reviewing the research; they are not yet a fresh-checkout, end-to-end workflow. The seven-contract audit and downloaded data used by notebook 05 live in Git-ignored `data/` and are not included in this commit.

Before running this downloader on a new machine, install `requirements-sandbox.txt` and generate or obtain the reviewed seven-contract audit. Do not bypass the seven-contract or rules-review checks. `RUN_DOWNLOADS = True` makes network requests; reviewing this notebook does not require executing it. The download is resumable, but do not run two copies against the same job directory.

Sources: [price-history API](https://docs.polymarket.com/api-reference/markets/get-a-tokens-price-history), [trade API](https://docs.polymarket.com/api-reference/feeds/list-trades), [machine-readable API specification](https://data-api.polymarket.com/v2/openapi.json).


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.error import HTTPError, URLError
from uuid import uuid4
import gzip
import hashlib
import json
import math
import os
import threading
import time

import numpy as np
import pandas as pd
from IPython.display import display
from polytrader.data.client import PolymarketClient, DataError
from polytrader.data.history import validate_point
from polytrader.research.storage import RateLimiter

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").exists() and (p / "src/polytrader").is_dir()), None
)
if ROOT is None:
    raise RuntimeError("Run inside the polytrader repository.")
STEP1 = ROOT / "data/research/deadline_markets/step1"
audit_pointer = json.loads((STEP1 / "latest.json").read_text(encoding="utf-8"))
AUDIT = ROOT / audit_pointer["run_path"]
contracts = pd.read_parquet(AUDIT / "contracts.parquet")
if len(contracts) != 7 or not contracts["rule_template_reviewed"].all():
    raise RuntimeError("Run the updated notebook 03 first: seven reviewed contracts required.")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)
print("Audit:", AUDIT)
display(contracts[["family", "deadline_date", "market_id", "api_start_date_utc", "closed_at_snapshot"]])


Audit: C:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader\data\research\deadline_markets\step1\runs\20261003T134840Z-627f6c2e


,family,deadline_date,market_id,api_start_date_utc,closed_at_snapshot
0,hormuz_normalisation,2026-09-30,2774057,2026-07-02 21:49:47.480763+00:00,False
1,hormuz_normalisation,2026-10-31,3501950,2026-08-11 09:19:18.848969+00:00,False
2,hormuz_normalisation,2026-11-30,3501951,2026-08-11 09:20:18.921762+00:00,False
3,hormuz_normalisation,2026-12-31,2176270,2026-05-11 13:03:57.905316+00:00,False
4,russia_ukraine_agreement,2026-10-31,2243896,2026-05-13 03:30:53.564000+00:00,False
5,russia_ukraine_agreement,2026-11-30,4338145,2026-09-07 15:51:23+00:00,False
6,russia_ukraine_agreement,2026-12-31,2243897,2026-05-13 03:30:47.258382+00:00,False


## 1. Controls and reproducible time window

`END_UTC = None` freezes the current time **once**, when this job is first created. A restart resumes that same snapshot; it does not silently move the end date. To collect a later snapshot, choose a new `JOB_NAME`.

Prices cover creation through that frozen time. All seven market-trade feeds are traversed using **v2 keyset cursors**. The market feed ignores start/end filters, so we filter timestamps locally. We do not use the old offset-limited trade downloader.

Rate: two requests per second across all workers, with bounded retries and `Retry-After` support. This is historical batch downloading, **not** polling a live market every second.


In [ ]:
RUN_DOWNLOADS = True
JOB_NAME = "sep_dec_2026_full_history_v1"
END_UTC = None                 # Optional ISO timestamp; cannot be in the future.
REQUESTS_PER_SECOND = 2.0
MAX_WORKERS = 3
MAX_ATTEMPTS = 5
TRADE_PAGE_SIZE = 1000
PRICE_PAGE_SIZE = 10000
# Nonzero tiny threshold: API's filter_amount=0 means a DEFAULT minimum of 0.01.
MIN_TRADE_SHARES = "0.000000000001"
# Overlapping retention probes. Empty fine-tier windows remain explicit.
PRICE_TIERS = [
    {"seconds": 60, "lookback_days": 14, "chunk_days": 1},
    {"seconds": 300, "lookback_days": 75, "chunk_days": 5},
    {"seconds": 1800, "lookback_days": 105, "chunk_days": 10},
    {"seconds": 10800, "lookback_days": None, "chunk_days": 15},
]
# Safety guard, not a target. Hitting it leaves a resumable INCOMPLETE job.
MAX_PAGES_PER_TASK = 20000
assert 0 < REQUESTS_PER_SECOND <= 4 and 1 <= MAX_WORKERS <= 4
assert JOB_NAME and all(c.isalnum() or c in "_-" for c in JOB_NAME)

BASE = ROOT / "data/research/deadline_markets"
JOB = BASE / "step2" / JOB_NAME
JOB.mkdir(parents=True, exist_ok=True)
LOCK = JOB / ".download.lock"
OUTPUT = JOB / "tables"
OUTPUT.mkdir(exist_ok=True)

def stable_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest(value):
    return hashlib.sha256(stable_json(value).encode("utf-8")).hexdigest()

def atomic_json(path, value):
    temp = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)

def atomic_parquet(path, frame):
    temp = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    frame.to_parquet(temp, index=False, compression="zstd")
    temp.replace(path)

identity = contracts[["market_id", "condition_id", "yes_token_id", "no_token_id", "rules_sha256"]].to_dict("records")
settings = {
    "schema_version": 1, "contracts": identity, "price_tiers": PRICE_TIERS,
    "trade_page_size": TRADE_PAGE_SIZE, "price_page_size": PRICE_PAGE_SIZE,
    "min_trade_shares": MIN_TRADE_SHARES,
}
config_path = JOB / "config.json"
if config_path.exists():
    config = json.loads(config_path.read_text(encoding="utf-8"))
    if config["settings_sha256"] != digest(settings):
        raise RuntimeError("Job settings or audited rules changed. Choose a new JOB_NAME.")
    if END_UTC is not None and pd.Timestamp(END_UTC).tz_convert("UTC").isoformat() != config["end_utc"]:
        raise RuntimeError("Different END_UTC: choose a new JOB_NAME.")
else:
    frozen_end = pd.Timestamp.now(tz="UTC").floor("s") if END_UTC is None else pd.Timestamp(END_UTC)
    if frozen_end.tzinfo is None:
        raise ValueError("END_UTC needs an explicit timezone, e.g. ...Z.")
    frozen_end = frozen_end.tz_convert("UTC").floor("s")
    if frozen_end > pd.Timestamp.now(tz="UTC"):
        raise ValueError("END_UTC cannot be in the future.")
    config = {
        "settings": settings, "settings_sha256": digest(settings),
        "end_utc": frozen_end.isoformat(),
        "audit_run": audit_pointer["run_path"],
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    atomic_json(config_path, config)
END_TS = int(pd.Timestamp(config["end_utc"]).timestamp())
print("Job:", JOB)
print("Frozen exclusive end:", config["end_utc"])


Job: C:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader\data\research\deadline_markets\step2\sep_dec_2026_full_history_v1
Frozen exclusive end: 2026-10-03T13:57:14+00:00


## 2. Download plan

Each token gets a coarse lifetime backbone plus overlapping finer windows. Querying one-second price buckets would not create one-second history and is not supported by this endpoint. We retain an empty response as a coverage fact, rather than confusing it with successful fine-resolution coverage.

Trade timestamps are locally restricted to the contract's creation through the frozen end. Post-deadline observations are saved but **flagged**, not silently included in a pre-deadline backtest.


In [ ]:
TRADES_URL = "https://data-api.polymarket.com/v2/trades"
PRICES_URL = "https://data-api.polymarket.com/v2/prices-history"
SPEC_URL = "https://data-api.polymarket.com/v2/openapi.json"
tasks = []
for row in contracts.to_dict("records"):
    start_ts = int(pd.Timestamp(row["market_created_at_utc"]).timestamp())
    if start_ts >= END_TS:
        raise ValueError("Frozen end precedes a selected contract's creation.")
    common = {
        "market_id": row["market_id"], "condition_id": row["condition_id"],
        "family": row["family"], "deadline_date": row["deadline_date"],
        "start_ts": start_ts, "end_ts": END_TS,
        "token_map": {row["yes_token_id"]: "Yes", row["no_token_id"]: "No"},
    }
    tasks.append({
        **common, "kind": "trades", "task_id": f"trades_{row['market_id']}",
        "url": TRADES_URL,
        "params": {"condition": row["condition_id"], "limit": TRADE_PAGE_SIZE,
                   "taker_only": "true", "filter_type": "TOKENS",
                   "filter_amount": MIN_TRADE_SHARES},
    })
    for token_id, outcome in common["token_map"].items():
        for tier in PRICE_TIERS:
            start = start_ts if tier["lookback_days"] is None else max(start_ts, END_TS - tier["lookback_days"] * 86400)
            while start < END_TS:
                end = min(start + tier["chunk_days"] * 86400, END_TS)
                tasks.append({
                    **common, "kind": "prices", "token_id": token_id, "outcome": outcome,
                    "start_ts": start, "end_ts": end, "requested_resolution_seconds": tier["seconds"],
                    "task_id": f"prices_{row['market_id']}_{outcome.lower()}_{tier['seconds']}_{start}_{end}",
                    "url": PRICES_URL,
                    "params": {"token_id": token_id, "start": start, "end": end,
                               "bucket_seconds": tier["seconds"], "limit": PRICE_PAGE_SIZE},
                })
                start = end
task_by_id = {task["task_id"]: task for task in tasks}
assert len(task_by_id) == len(tasks)
atomic_json(JOB / "plan.json", tasks)
display(pd.DataFrame(tasks).groupby("kind").size().rename("tasks").to_frame())
print("Price tasks may need extra pages; trade-feed lengths are not known in advance.")
print("Minimum request spacing:", 1 / REQUESTS_PER_SECOND, "seconds (shared across workers).")


,tasks
kind,
prices,582
trades,7


Price tasks may need extra pages; trade-feed lengths are not known in advance.
Minimum request spacing: 0.5 seconds (shared across workers).


## 3. Resumable public-data downloader

Every page is durably saved **before** advancing its cursor. Interruptions reuse saved pages. A repeated/missing cursor, wrong condition/token, invalid price or invalid timestamp fails that task visibly. No transaction-hash deduplication is applied: one transaction can contain several legitimate fills, including identical-looking fills.

Trade pages retain price, size, side, token/condition IDs, timestamp and transaction hash; wallet/profile fields are discarded. Page coordinates provide a source-record key, not an invented exchange trade ID.

The job lock prevents concurrent notebook copies. If Python was forcibly killed and left a lock file, confirm the old process is no longer running before removing only that lock.


In [ ]:
limiter = RateLimiter(REQUESTS_PER_SECOND)
log_lock = threading.Lock()
stop_event = threading.Event()

def log(message):
    with log_lock:
        print(message, flush=True)

def request_json(url, params):
    for attempt in range(1, MAX_ATTEMPTS + 1):
        if stop_event.is_set():
            raise InterruptedError("Download stopped; saved pages are reusable.")
        limiter.wait()
        try:
            return PolymarketClient(timeout=30).get_json(url, **params)
        except DataError as exc:
            cause = exc.__cause__
            retryable = (
                isinstance(cause, HTTPError) and (cause.code in {408, 429} or cause.code >= 500)
                or isinstance(cause, (URLError, TimeoutError, ConnectionError, OSError))
            )
            if not retryable or attempt == MAX_ATTEMPTS:
                raise
            delay = min(30, 2 ** attempt)
            if isinstance(cause, HTTPError):
                retry_after = cause.headers.get("Retry-After")
                if retry_after:
                    try:
                        delay = max(delay, float(retry_after))
                    except ValueError:
                        from email.utils import parsedate_to_datetime
                        try:
                            delay = max(delay, (parsedate_to_datetime(retry_after) - datetime.now(timezone.utc)).total_seconds())
                        except (TypeError, ValueError):
                            pass
            log(f"Retry {attempt}/{MAX_ATTEMPTS} in {delay:.1f}s: {exc}")
            if stop_event.wait(delay):
                raise InterruptedError("Stopped during retry delay.")

TRADE_FIELDS = [
    "timestamp", "condition_id", "token_id", "side", "price", "size",
    "outcome", "outcome_index", "transaction_hash",
]

def checked_payload(task, page):
    if not isinstance(page, dict) or not isinstance(page.get("data"), list):
        raise ValueError("API response has no data array.")
    pagination = page.get("pagination")
    if not isinstance(pagination, dict) or type(pagination.get("has_more")) is not bool:
        raise ValueError("API response has no valid pagination.")
    if pagination["has_more"] and (not isinstance(pagination.get("next_cursor"), str) or not pagination["next_cursor"]):
        raise ValueError("API claims another page without a usable cursor.")
    if pagination["has_more"] and not page["data"]:
        raise ValueError("Empty page claims another page; refusing an ambiguous walk.")
    previous_timestamp = None
    for row in page["data"]:
        if task["kind"] == "prices":
            validate_point(row)
            if not task["start_ts"] <= row["timestamp"] <= task["end_ts"]:
                raise ValueError("Price observation outside requested window.")
            if row["timestamp"] == task["end_ts"] and row["resolution_seconds"] != 0:
                raise ValueError("Non-settlement price at exclusive end.")
            if previous_timestamp is not None and row["timestamp"] < previous_timestamp:
                raise ValueError("Price page not oldest-first.")
        else:
            if not isinstance(row, dict) or type(row.get("timestamp")) is not int:
                raise ValueError("Invalid trade timestamp.")
            if row.get("condition_id") != task["condition_id"]:
                raise ValueError("Wrong market returned by the trade filter.")
            if row.get("token_id") not in task["token_map"]:
                raise ValueError("Unknown outcome token returned.")
            if row.get("side") not in {"BUY", "SELL"}:
                raise ValueError("Unknown trade side.")
            if (type(row.get("price")) not in {int, float} or not math.isfinite(row["price"])
                or not 0 <= row["price"] <= 1
                or type(row.get("size")) not in {int, float} or not math.isfinite(row["size"])
                or row["size"] <= 0):
                raise ValueError("Invalid trade price or size.")
            if previous_timestamp is not None and row["timestamp"] > previous_timestamp:
                raise ValueError("Trade page not newest-first.")
        previous_timestamp = row["timestamp"]
    rows = page["data"] if task["kind"] == "prices" else [
        {field: row.get(field) for field in TRADE_FIELDS} for row in page["data"]
    ]
    return {"data": rows, "pagination": pagination}

def write_page(path, payload):
    temp = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    with gzip.open(temp, "wt", encoding="utf-8") as sink:
        json.dump(payload, sink, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    temp.replace(path)

def read_page(path):
    with gzip.open(path, "rt", encoding="utf-8") as source:
        return json.load(source)

def download_task(task):
    folder = JOB / "pages" / task["task_id"]
    folder.mkdir(parents=True, exist_ok=True)
    task_hash = digest(task)
    state_path = folder / "state.json"
    state = json.loads(state_path.read_text()) if state_path.exists() else {
        "task_id": task["task_id"], "task_sha256": task_hash,
        "status": "pending", "next_page": 0, "next_cursor": None,
        "rows_received": 0, "last_timestamp": None,
    }
    if state["task_sha256"] != task_hash:
        raise ValueError("Task parameters changed; use a new job.")
    if state["status"] == "complete":
        return state
    seen_cursors = set()
    try:
        while state["next_page"] < MAX_PAGES_PER_TASK:
            if stop_event.is_set():
                raise InterruptedError("Interrupted.")
            index = state["next_page"]
            cursor = state["next_cursor"]
            if cursor is not None and cursor in seen_cursors:
                raise ValueError("Repeated pagination cursor.")
            seen_cursors.add(cursor)
            path = folder / f"page_{index:06d}.json.gz"
            if path.exists():
                saved = read_page(path)
                if saved["task_sha256"] != task_hash or saved["request_cursor"] != cursor:
                    raise ValueError("Saved page belongs to a different request.")
                payload = checked_payload(task, saved["response"])
            else:
                params = {**task["params"], "cursor": cursor}
                response = request_json(task["url"], params)
                payload = checked_payload(task, response)
                saved = {
                    "task_sha256": task_hash, "request_cursor": cursor,
                    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
                    "response_sha256_before_field_selection": digest(response),
                    "response": payload,
                }
                write_page(path, saved)
            rows = payload["data"]
            previous = state["last_timestamp"]
            if previous is not None and rows:
                first = rows[0]["timestamp"]
                if (task["kind"] == "trades" and first > previous
                    or task["kind"] == "prices" and first < previous):
                    raise ValueError("Pagination reverses chronological order.")
            pagination = payload["pagination"]
            next_cursor = pagination.get("next_cursor")
            if pagination["has_more"] and (next_cursor == cursor or next_cursor in seen_cursors):
                raise ValueError("Pagination cursor did not advance.")
            state.update({
                "status": "running" if pagination["has_more"] else "complete",
                "next_page": index + 1, "next_cursor": next_cursor if pagination["has_more"] else None,
                "rows_received": state["rows_received"] + len(rows),
                "last_timestamp": rows[-1]["timestamp"] if rows else previous,
                "updated_at_utc": datetime.now(timezone.utc).isoformat(),
                "error": None,
            })
            atomic_json(state_path, state)
            if task["kind"] == "trades" and ((index + 1) % 10 == 0 or state["status"] == "complete"):
                log(f"Trades {task['market_id']}: {state['rows_received']:,} rows; {state['status']}")
            if state["status"] == "complete":
                return state
        raise RuntimeError("Page safety limit reached; increase explicitly and rerun.")
    except Exception as exc:
        state.update({"status": "interrupted" if isinstance(exc, InterruptedError) else "failed",
                      "error": f"{type(exc).__name__}: {exc}",
                      "updated_at_utc": datetime.now(timezone.utc).isoformat()})
        atomic_json(state_path, state)
        raise


### Offline safety checks

Before requesting history, verify reversed outcome order, invalid trade records, cursor failures and price-window boundaries. Checks below use synthetic records only.


In [ ]:
trade_task = next(t for t in tasks if t["kind"] == "trades")
price_task = next(t for t in tasks if t["kind"] == "prices")
test_token = next(iter(trade_task["token_map"]))
trade_row = {"timestamp": 123, "condition_id": trade_task["condition_id"],
             "token_id": test_token, "side": "BUY", "size": 2.0, "price": 0.5}
last_page = {"data": [trade_row], "pagination": {"has_more": False, "next_cursor": None}}
assert checked_payload(trade_task, last_page)["data"][0]["size"] == 2.0
def must_reject(task, payload):
    try:
        checked_payload(task, payload)
    except (ValueError, DataError):
        return
    raise AssertionError("Invalid API payload accepted")
must_reject(trade_task, {"data": [dict(trade_row, condition_id="wrong")], "pagination": last_page["pagination"]})
must_reject(trade_task, {"data": [dict(trade_row, size=-1)], "pagination": last_page["pagination"]})
must_reject(trade_task, {"data": [trade_row], "pagination": {"has_more": True, "next_cursor": None}})
must_reject(price_task, {"data": [{"timestamp": price_task["end_ts"], "price": 0.4, "resolution_seconds": 60}],
                         "pagination": last_page["pagination"]})
assert checked_payload(price_task, {
    "data": [{"timestamp": price_task["end_ts"], "price": 0, "resolution_seconds": 0}],
    "pagination": last_page["pagination"],
})["data"][0]["resolution_seconds"] == 0
print("Passed API schema, pagination, market filtering and exclusive-window checks.")


Passed API schema, pagination, market filtering and exclusive-window checks.


In [ ]:
if RUN_DOWNLOADS:
    stop_event.clear()
    try:
        lock_fd = os.open(LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError:
        raise RuntimeError(f"Another run or a stale lock exists: {LOCK}. Inspect before removing.")
    with os.fdopen(lock_fd, "w", encoding="utf-8") as sink:
        sink.write(json.dumps({"pid": os.getpid(), "started_at_utc": datetime.now(timezone.utc).isoformat()}))
    results, failures = [], []
    executor = None
    try:
        if not (JOB / "api_spec.json").exists():
            atomic_json(JOB / "api_spec.json", request_json(SPEC_URL, {}))
        # Save the exact audit alongside the data job, independent of latest.json.
        if not (JOB / "contracts.parquet").exists():
            atomic_parquet(JOB / "contracts.parquet", contracts)
        executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)
        # Small feeds first; price tasks submitted alongside larger feed walks.
        schedule = sorted(tasks, key=lambda task: (
            0 if task["kind"] == "trades" and task["family"] == "russia_ukraine_agreement" else 1,
            0 if task["kind"] == "prices" else 1,
            task["task_id"],
        ))
        futures = {executor.submit(download_task, task): task for task in schedule}
        for future in as_completed(futures):
            task = futures[future]
            try:
                results.append(future.result())
            except Exception as exc:
                failures.append({"task_id": task["task_id"], "error": str(exc)})
                log(f"FAILED {task['task_id']}: {exc}")
            progress = {
                "tasks_total": len(tasks), "tasks_finished": len(results) + len(failures),
                "tasks_complete": len(results), "tasks_failed": len(failures),
                "rows_in_completed_tasks": sum(r["rows_received"] for r in results),
                "updated_at_utc": datetime.now(timezone.utc).isoformat(),
                "failures": failures,
            }
            atomic_json(JOB / "progress.json", progress)
            if progress["tasks_finished"] % 25 == 0 or progress["tasks_finished"] == len(tasks):
                log(f"Tasks {progress['tasks_finished']}/{len(tasks)}; failures {len(failures)}")
    finally:
        stop_event.set()
        if executor is not None:
            executor.shutdown(wait=True, cancel_futures=True)
        LOCK.unlink(missing_ok=True)  # Only the exact lock this run created.
    if failures:
        print("Some tasks failed. Run All to retry; successful pages are preserved.")
else:
    print("Offline mode: using downloaded pages only.")


Trades 4338145: 196 rows; complete


Trades 2243897: 10,000 rows; running


Trades 2243896: 10,000 rows; running


Trades 2243896: 11,309 rows; complete


Tasks 25/589; failures 0


...

Trades 2774057: 16,161 rows; complete


Trades 2176270: 20,000 rows; running


Trades 2176270: 30,000 rows; running


Trades 2176270: 33,912 rows; complete


Tasks 589/589; failures 0


## 4. Build analysis tables without inventing observations

- `trades.parquet`: individual returned fills, keeping same-second and same-transaction fills separate.
- `prices.parquet`: native-resolution observations (multiple resolutions intentionally coexist).
- `terminal_price_points.parquet`: resolution-zero points kept separately; never used as ordinary trading prices or as independently verified resolution times.
- `trade_bars_1s.parquet` and `trade_bars_1min.parquet`: **sparse** intervals with actual trades, per outcome. VWAP, share volume, price-times-size notional and trade counts; no forward-filling and no manufactured zero-volume intervals.

Bar timestamps are **interval ends**. Thus a trade at 12:00:00 belongs to [12:00:00,12:00:01) and its one-second bar is labeled 12:00:01. This still does not establish when an exchange quote or a trader could have observed the trade.

Hormuz has no precise intraday cutoff in its observation rule: its `pre_deadline_proxy` uses API endDate as an explicitly labeled approximation. Use the saved rules when defining the final backtest. An expired deadline is not necessarily a resolved market.


In [ ]:
states = []
price_records, trade_records, terminal_records = [], [], []
for task in tasks:
    folder = JOB / "pages" / task["task_id"]
    state_path = folder / "state.json"
    state = json.loads(state_path.read_text()) if state_path.exists() else {
        "status": "not_started", "next_page": 0, "rows_received": 0, "error": None,
    }
    states.append({
        "task_id": task["task_id"], "kind": task["kind"], "market_id": task["market_id"],
        "token_id": task.get("token_id"), "requested_resolution_seconds": task.get("requested_resolution_seconds"),
        "start_ts": task["start_ts"], "end_ts": task["end_ts"],
        "status": state["status"], "pages": state["next_page"],
        "rows_received": state["rows_received"], "error": state.get("error"),
    })
    # Only committed pages; an orphan saved just before interruption is reused on resume.
    for page_index in range(state["next_page"]):
        saved = read_page(folder / f"page_{page_index:06d}.json.gz")
        if saved["task_sha256"] != digest(task):
            raise ValueError("Saved page/task mismatch.")
        payload = checked_payload(task, saved["response"])
        for row_index, row in enumerate(payload["data"]):
            common = {
                "family": task["family"], "deadline_date": task["deadline_date"],
                "market_id": task["market_id"], "condition_id": task["condition_id"],
                "source_task": task["task_id"], "source_page": page_index, "source_row": row_index,
            }
            if task["kind"] == "prices":
                record = {**common, **row, "token_id": task["token_id"], "outcome": task["outcome"],
                          "requested_resolution_seconds": task["requested_resolution_seconds"],
                          "retrieved_at_utc": saved["retrieved_at_utc"]}
                if row["resolution_seconds"] == 0:
                    terminal_records.append(record)
                elif task["start_ts"] <= row["timestamp"] < task["end_ts"]:
                    price_records.append(record)
            elif task["start_ts"] <= row["timestamp"] < task["end_ts"]:
                trade_records.append({
                    **common, **row, "outcome": task["token_map"][row["token_id"]],
                    "source_record_id": f"{task['task_id']}:{page_index}:{row_index}",
                })
coverage = pd.DataFrame(states)
prices = pd.DataFrame(price_records)
trades = pd.DataFrame(trade_records)
terminal_points = pd.DataFrame(terminal_records)
if prices.empty or trades.empty:
    raise RuntimeError("No usable prices or trades yet. Inspect progress and resume before building tables.")

# Overlapping price tiers may repeat a timestamp/resolution. Preserve disagreements
# separately; retain the most recently retrieved version, without averaging prices.
price_keys = ["token_id", "timestamp", "resolution_seconds"]
conflict_keys = prices.groupby(price_keys)["price"].nunique()
conflict_keys = conflict_keys[conflict_keys > 1].reset_index()[price_keys]
price_conflicts = prices.merge(conflict_keys, on=price_keys, how="inner")
prices = prices.sort_values("retrieved_at_utc").drop_duplicates(price_keys, keep="last")
prices = prices.sort_values(["token_id", "timestamp", "resolution_seconds"]).reset_index(drop=True)
trades = trades.sort_values(["token_id", "timestamp", "source_page", "source_row"]).reset_index(drop=True)
assert trades["source_record_id"].is_unique
assert ((prices.price >= 0) & (prices.price <= 1)).all()
assert prices.resolution_seconds.gt(0).all()
assert trades["size"].gt(0).all()
assert set(trades.token_id).issubset(set(contracts.yes_token_id) | set(contracts.no_token_id))
trades["notional_price_times_size"] = trades["price"] * trades["size"]

cutoffs = {}
cutoff_basis = {}
for row in contracts.to_dict("records"):
    explicit = row["rule_cutoff_utc"]
    cutoff = row["api_end_date_utc"] if pd.isna(explicit) else explicit
    cutoffs[row["market_id"]] = int(pd.Timestamp(cutoff).timestamp())
    cutoff_basis[row["market_id"]] = "API endDate proxy; rule is date-based" if pd.isna(explicit) else "explicit rule cutoff"
for frame in [prices, trades]:
    frame["timestamp_utc"] = pd.to_datetime(frame["timestamp"], unit="s", utc=True)
    frame["pre_deadline_proxy"] = frame["timestamp"] < frame["market_id"].map(cutoffs)
    frame["cutoff_basis"] = frame["market_id"].map(cutoff_basis)

def trade_bars(frame, seconds):
    work = frame.assign(bar_end=(frame["timestamp"] // seconds + 1) * seconds)
    bars = work.groupby(["family", "deadline_date", "market_id", "token_id", "outcome", "bar_end"], sort=True).agg(
        trade_count=("size", "size"), volume_shares=("size", "sum"),
        notional_price_times_size=("notional_price_times_size", "sum"),
        low=("price", "min"), high=("price", "max"),
        first_trade_timestamp=("timestamp", "min"), last_trade_timestamp=("timestamp", "max"),
    ).reset_index()
    bars["vwap"] = bars["notional_price_times_size"] / bars["volume_shares"]
    bars["bar_end_utc"] = pd.to_datetime(bars["bar_end"], unit="s", utc=True)
    bars["interval_seconds"] = seconds
    bars["whole_bar_pre_deadline_proxy"] = bars["bar_end"] <= bars["market_id"].map(cutoffs)
    bars["cutoff_basis"] = bars["market_id"].map(cutoff_basis)
    return bars

bars_1s = trade_bars(trades, 1)
bars_1min = trade_bars(trades, 60)
assert bars_1s.trade_count.sum() == len(trades) == bars_1min.trade_count.sum()
assert np.isclose(bars_1s.volume_shares.sum(), trades["size"].sum())
assert np.isclose(bars_1min.notional_price_times_size.sum(), trades.notional_price_times_size.sum())
# Boundary test: a trade exactly on a minute belongs to the next ending minute.
assert (120 // 60 + 1) * 60 == 180
assert (120 // 1 + 1) * 1 == 121

for name, frame in {
    "prices": prices, "trades": trades, "terminal_price_points": terminal_points,
    "price_conflicts": price_conflicts, "task_coverage": coverage,
    "trade_bars_1s": bars_1s, "trade_bars_1min": bars_1min,
}.items():
    atomic_parquet(OUTPUT / f"{name}.parquet", frame)
print(f"Saved {len(prices):,} prices, {len(trades):,} trades, {len(bars_1s):,} occupied one-second outcome intervals.")
print(f"Separately retained {len(terminal_points):,} resolution-zero points and {len(price_conflicts):,} conflicting price records.")


Saved 416,372 prices, 96,092 trades, 70,258 occupied one-second outcome intervals.
Separately retained 212 resolution-zero points and 0 conflicting price records.


## 5. Coverage and manifest — read this before backtesting

“Completed” below means every planned API cursor walk finished, **not** that Polymarket guarantees all historical activity is retained, all chart points are fresh quotes, or all missing intervals had no trading.

Historical fine-tier retention is probed only over the configured lookbacks. Expired one-minute history cannot be recreated by changing a parameter. Individual trades can give finer event-time observations, but not a historical order book or executable bid/ask prices.

The frozen end is a collection boundary, not a finalized blockchain snapshot; delayed indexing/reorganizations are not independently verified here. No claim of a profitable strategy is made.


In [ ]:
market_summary = contracts[["family", "deadline_date", "market_id"]].copy()
trade_summary = trades.groupby("market_id").agg(
    trades=("timestamp", "size"), first_trade=("timestamp_utc", "min"),
    last_trade=("timestamp_utc", "max"), volume_shares=("size", "sum"),
    notional_price_times_size=("notional_price_times_size", "sum"),
    pre_deadline_trades=("pre_deadline_proxy", "sum"),
).reset_index()
price_summary = prices.groupby("market_id").agg(
    price_points=("timestamp", "size"), first_price=("timestamp_utc", "min"),
    last_price=("timestamp_utc", "max"),
).reset_index()
market_summary = market_summary.merge(trade_summary, on="market_id", how="left").merge(price_summary, on="market_id", how="left")
resolutions = prices.groupby(["family", "deadline_date", "outcome", "resolution_seconds"]).agg(
    observations=("timestamp", "size"), first=("timestamp_utc", "min"),
    last=("timestamp_utc", "max"),
).reset_index()
token_summary = prices.groupby(["market_id", "token_id", "outcome"]).agg(
    points=("timestamp", "size"), distinct_seconds=("timestamp", "nunique"),
    first=("timestamp_utc", "min"), last=("timestamp_utc", "max"),
).reset_index()
expected_tokens = pd.DataFrame([
    {"market_id": row["market_id"], "token_id": row[f"{label.lower()}_token_id"], "outcome": label}
    for row in contracts.to_dict("records") for label in ["Yes", "No"]
])
token_summary = expected_tokens.merge(token_summary, how="left", on=["market_id", "token_id", "outcome"])
for name, frame in {"market_summary": market_summary, "resolutions": resolutions, "token_coverage": token_summary}.items():
    atomic_parquet(OUTPUT / f"{name}.parquet", frame)
    frame.to_csv(OUTPUT / f"{name}.csv", index=False, encoding="utf-8-sig")

complete = bool(coverage.status.eq("complete").all())
artifact_info = {
    path.name: {"bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    for path in sorted(OUTPUT.iterdir()) if path.is_file() and not path.name.endswith(".tmp")
}
manifest = {
    "schema_version": 1, "job": JOB_NAME, "frozen_end_utc": config["end_utc"],
    "audit_run": config["audit_run"], "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete_api_walks" if complete else "partial",
    "contract_count": len(contracts), "expected_outcome_tokens": len(expected_tokens),
    "tasks_total": len(tasks), "tasks_complete": int(coverage.status.eq("complete").sum()),
    "tasks_with_empty_response": int(coverage.rows_received.eq(0).sum()),
    "rows": {"prices": len(prices), "trades": len(trades), "one_second_bars": len(bars_1s),
             "one_minute_bars": len(bars_1min), "resolution_zero_points": len(terminal_points),
             "conflicting_price_records": len(price_conflicts)},
    "sources": [TRADES_URL, PRICES_URL, SPEC_URL],
    "parameters": settings,
    "limitations": [
        "Ukraine September 2026 agreement contract not found; no substitute used.",
        "Trade timestamps are blockchain seconds, not exchange execution/quote availability times.",
        "Taker-only public trade rows above the configured tiny positive size filter; API availability not independently certified.",
        "Market trade feed uses a fixed three-year window and ignores start/end; normalized records are filtered locally.",
        "One-second bars are sparse trade aggregates, not continuous quotes.",
        "No interpolation, price forward-fill, invented zero-volume intervals or same-hash fill deduplication.",
        "Price records retain their actual API resolution; fine-tier retention is limited.",
        "Resolution-zero price points are quarantined, not ground-truth resolution timestamps.",
        "Hormuz uses an explicitly flagged API endDate cutoff proxy; final backtest must consider publication rules.",
        "Contemporary rule snapshots cannot prove historical rule wording.",
        "No historical bid/ask, executable profitability or out-of-sample strategy validation performed.",
    ],
    "artifacts": artifact_info,
}
atomic_json(JOB / "manifest.json", manifest)
atomic_json(BASE / "step2/latest.json", {"job": JOB_NAME, "job_path": JOB.relative_to(ROOT).as_posix()})
display(market_summary)
display(resolutions)
print("Status:", manifest["status"])
print("Manifest:", JOB / "manifest.json")
print("Tables:", OUTPUT)
if not complete:
    display(coverage.loc[~coverage.status.eq("complete"), ["task_id", "status", "error"]])
    print("INCOMPLETE: rerun the download cell before interpreting missing data.")


,family,deadline_date,market_id,trades,first_trade,last_trade,volume_shares,notional_price_times_size,pre_deadline_trades,price_points,first_price,last_price
0,hormuz_normalisation,2026-09-30,2774057,16161,2026-07-02 23:37:50+00:00,2026-10-03 13:55:48+00:00,1.229433e+07,4.808467e+06,16140,64205,2026-07-03 00:00:00+00:00,2026-10-03 13:55:00+00:00
1,hormuz_normalisation,2026-10-31,3501950,2430,2026-08-11 11:50:09+00:00,2026-10-03 12:49:17+00:00,1.707523e+06,5.148846e+05,2430,59039,2026-08-11 09:00:00+00:00,2026-10-03 13:55:00+00:00
2,hormuz_normalisation,2026-11-30,3501951,1399,2026-08-11 11:50:16+00:00,2026-10-03 12:54:05+00:00,5.518404e+05,1.809452e+05,1399,59104,2026-08-11 09:00:00+00:00,2026-10-03 13:55:00+00:00
3,hormuz_normalisation,2026-12-31,2176270,33911,2026-05-11 13:30:06+00:00,2026-10-03 13:56:36+00:00,1.392387e+07,6.211361e+06,33911,64557,2026-05-12 00:00:00+00:00,2026-10-03 13:52:00+00:00
4,russia_ukraine_agreement,2026-10-31,2243896,11309,2026-05-13 04:27:15+00:00,2026-10-03 13:13:30+00:00,1.831239e+06,7.724907e+05,11309,64458,2026-05-13 12:00:00+00:00,2026-10-03 13:55:00+00:00
5,russia_ukraine_agreement,2026-11-30,4338145,196,2026-09-07 16:46:21+00:00,2026-10-03 12:05:30+00:00,3.877914e+04,1.673535e+04,196,40438,2026-09-07 15:55:00+00:00,2026-10-03 13:55:00+00:00
6,russia_ukraine_agreement,2026-12-31,2243897,30686,2026-05-13 04:44:19+00:00,2026-10-03 12:54:47+00:00,2.729607e+06,1.352694e+06,30686,64571,2026-05-13 12:00:00+00:00,2026-10-03 13:55:00+00:00


,family,deadline_date,outcome,resolution_seconds,observations,first,last
0,hormuz_normalisation,2026-09-30,No,60,11327,2026-09-25 17:00:00+00:00,2026-10-03 13:55:00+00:00
1,hormuz_normalisation,2026-09-30,No,300,16965,2026-07-30 00:00:00+00:00,2026-10-03 13:45:00+00:00
2,hormuz_normalisation,2026-09-30,No,1800,3079,2026-07-05 00:00:00+00:00,2026-10-03 12:30:00+00:00
3,hormuz_normalisation,2026-09-30,No,10800,708,2026-07-03 00:00:00+00:00,2026-10-03 06:00:00+00:00
4,hormuz_normalisation,2026-09-30,Yes,60,11327,2026-09-25 17:00:00+00:00,2026-10-03 13:55:00+00:00
5,hormuz_normalisation,2026-09-30,Yes,300,17002,2026-07-30 00:00:00+00:00,2026-10-03 13:45:00+00:00
6,hormuz_normalisation,2026-09-30,Yes,1800,3084,2026-07-05 00:00:00+00:00,2026-10-03 12:30:00+00:00
7,hormuz_normalisation,2026-09-30,Yes,10800,713,2026-07-03 00:00:00+00:00,2026-10-03 06:00:00+00:00
8,hormuz_normalisation,2026-10-31,No,60,11328,2026-09-25 17:00:00+00:00,2026-10-03 13:55:00+00:00
9,hormuz_normalisation,2026-10-31,No,300,15191,2026-08-11 09:15:00+00:00,2026-10-03 13:45:00+00:00


Status: complete_api_walks
Manifest: C:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader\data\research\deadline_markets\step2\sep_dec_2026_full_history_v1\manifest.json
Tables: C:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader\data\research\deadline_markets\step2\sep_dec_2026_full_history_v1\tables


## Next: choose a defensible backtest window

For September Hormuz, restrict the first test to observations before its deadline proxy and examine the publication clauses. Never let October observations or resolution-zero points predict September.

Compare synchronized **observed price changes**, report stale prices and missing intervals, and test held-out dates. The one-second trade table is useful for inspecting bursts of activity, but apparent same-second sequencing is not reliable evidence that a trader could react before another contract adjusted.

To rerun analysis offline: set `RUN_DOWNLOADS = False`. To resume an interrupted download: leave it True and keep the same `JOB_NAME`. The notebook does not start continuous live monitoring.
